In [ ]:
## merge data files into a single dataframe
import os
import pandas as pd
dir_path = 'rent_data/'
file_paths = os.listdir(dir_path)
data = []
for file_path in file_paths:
    file_path = dir_path+file_path
    data_part = pd.read_csv(file_path)
    data.append(data_part)
data = pd.concat(data)

In [ ]:
## data overview
print(data.info())
data.head()

In [ ]:
data['??'].unique()

Looks we have outliers for ?? column.

## Cleansing
cleansing target:
1. missing values
3. outliers: price, area, category
4. floor column: split -> floor | total floors
5. subway column: split -> subway line | distance to station
5. data type: Price, Area, Floor Total Floor, Number of Subways, Distance to Station

In [ ]:
## missing values
data.dropna(axis=0,inplace=True)
data.info()

In [ ]:
## remove duplicates
data.drop_duplicates(inplace=True)

In [ ]:
col_name_mapping = {
    '??':'Price',
    '??':'Area',
    '??':'Id',
    '??':'House Type',
    '??':'FloorInfo',
    '??1':'District',
    '??2':'Address',
    '??':'Community',
    '??':'Subway',
}
data.rename(columns=col_name_mapping,inplace=True)
data.head()

In [ ]:
## outliers
data = data[data['Price'] != '??']
data = data[data['Area'] != '??']
data = data[data['House Type'] != '??']
data.info()


In [ ]:
import numpy
data.loc[:,'Floor'] = data['FloorInfo'].apply(lambda x: int(x.split('/')[0]))
data.loc[:,'Total Floors'] = data['FloorInfo'].apply(lambda x: int(x.split('/')[-1].replace('?','')))
data.index = range(len(data))
indices_to_swap = data[data['Floor'] > data['Total Floors']].index
data.loc[indices_to_swap, ['Floor', 'Total Floors']] = data.loc[indices_to_swap, ['Total Floors', 'Floor']].to_numpy()


data.loc[:,'Number of Subways'] = data['Subway'].str.count('?')
data['Distance to Subway'] = data['Subway'].str.extract(r'(\d+)?')
data['Distance to Subway'].fillna(-1,inplace=True)
data.drop(columns=['FloorInfo','Subway'],inplace=True)
data.loc[[2150]]

In [ ]:
data['Price'] = data['Price'].astype(numpy.int64)
data['Area'] = data['Area'].astype(numpy.int64)
data['Number of Subways'] = data['Number of Subways'].astype(numpy.int64)
data['Distance to Subway'] = data['Distance to Subway'].astype(numpy.int64)


In [ ]:
data.info()

In [ ]:
data.head()

In [ ]:
data.to_csv('cleaned_rent.csv',index=False)

In [ ]:
from pyecharts import options
from pyecharts.charts import *
s = data.groupby('District').size().sort_values(ascending=True).to_dict()
house_number_bar = Bar()
house_number_bar.add_xaxis(list(s.keys()))
house_number_bar.add_yaxis('District',list(s.values()),color='grey')
house_number_bar.reversal_axis()
house_number_bar.set_global_opts(
    title_opts=options.TitleOpts(title='Number of House Segemented by Districts'),
)
house_number_bar.set_series_opts(
    label_opts=options.LabelOpts(position='right')
)
house_number_bar.render_notebook()

In [ ]:
s = data.groupby('Community').apply(lambda x: int(x['Price'].sum()/x['Area'].sum())).sort_values(ascending=True)[-10:].to_dict()
rent_average_bar = Bar()
rent_average_bar.add_xaxis(list(s.keys()))
rent_average_bar.add_yaxis('average rental fee',list(s.values()),color='grey')
rent_average_bar.reversal_axis()
rent_average_bar = Grid().add(rent_average_bar,grid_opts=options.GridOpts(pos_left='15%'))
rent_average_bar.render_notebook()

In [ ]:
print(data.head())
if 'Number of Rooms' not in data.columns:
    data.loc[:,'Number of Rooms'] = data['House Type'].str.extract(r'(\d+?)')
s = data.groupby('Number of Rooms').size().sort_values().to_dict()
house_number_rooms = Bar()
house_number_rooms.add_xaxis(list(s.keys()))
house_number_rooms.add_yaxis('number of rooms',list(s.values()),color='grey')
house_number_rooms.reversal_axis()
house_number_rooms.set_series_opts(
    label_opts=options.LabelOpts(position='right')
)
house_number_rooms.render_notebook()

In [ ]:
data['Has Lift'] = data['Total Floors'] > 7

values = data['Has Lift'].value_counts(normalize=False)
categories = ['has lift', 'has no lift']
pairs = [list(z) for z in zip(categories,values)]

pie_chart = (
    Pie()
    .add(
        '',
        data_pair=pairs,
        label_opts=options.LabelOpts(
            formatter='{d}%',
            position='inside',
            font_size=18
        )
    )
)

pie_chart.render_notebook()


In [ ]:
floor_category_label=['Low','Medium','High']
floor_category_bin=[0,0.33,0.66,1]
data['Floor Category'] = pd.cut(data['Floor']/data['Total Floors'], bins=floor_category_bin, labels=floor_category_label).astype(str)
data.loc[data['Has Lift'],'Floor Category'] += ' with Lift'
data.loc[~data['Has Lift'],'Floor Category'] += ' without Lift'

floor_category_freq_dict = data.groupby('Floor Category').size().to_dict()
floor_category_avg_rent_dict = data.groupby('Floor Category').agg({'Price':'sum','Area':'sum'})
floor_category_avg_rent_dict['Rental Fee Per Area'] = floor_category_avg_rent_dict['Price']/floor_category_avg_rent_dict['Area']
floor_category_avg_rent_dict = floor_category_avg_rent_dict['Rental Fee Per Area'].round(0).astype(int).to_dict()

line_chart = (
    Line()
    .add_xaxis(list(floor_category_freq_dict.keys()))
    .add_yaxis("number of rooms",list(floor_category_freq_dict.values()))
    .add_yaxis("rental fee per meter square",list(floor_category_avg_rent_dict.values()))
)

line_chart.render_notebook()


## Commute

In [ ]:
n_subways_size_dict = data.groupby('Number of Subways').agg({'Price':'sum','Area':'sum'})
n_subways_size_dict['Rental Fee'] = n_subways_size_dict['Price']/n_subways_size_dict['Area']
n_subways_size_dict = n_subways_size_dict['Rental Fee'].round(0).astype(int).to_dict()

n_subways_size_chart = (
    Line()
    .add_xaxis(list(n_subways_size_dict.keys()))
    .add_yaxis('Number of Rooms',list(n_subways_size_dict.values()))
)
n_subways_size_chart.render_notebook()

In [ ]:
dis_category = ['<500 m','500-1000 m','1000-1500 m','1500+ m']
dis_bins = [0,500,1000,1500,numpy.inf]
data['Distance Category'] = pd.cut(data['Distance to Subway'],bins=dis_bins,labels=dis_category).astype(str)
dis_rental = data.groupby('Distance Category').agg({'Price':'sum','Area':'sum'})
dis_rental['Rental'] = dis_rental['Price']/dis_rental['Area']
dis_rental = dis_rental['Rental'].dropna().round(0).astype(int).sort_values(ascending=False).to_dict()
dis_rental.pop('nan',None)

dis_rental_chart = (
    Line()
    .add_xaxis(list(dis_rental.keys()))
    .add_yaxis('',list(dis_rental.values()))
)
dis_rental_chart.render_notebook()